<h1>데이터 로드</h1>

In [1]:
from google.colab import drive
import os

# 1. 구글 드라이브 마운트 (권한 허용 필요)
drive.mount('/content/drive')

# 2. 드라이브의 zip 파일을 코랩 임시 디스크로 압축 해제 (채점용 데이터)
print("데이터 압축을 푸는 중입니다... (약 1~2분 소요)")
!unzip -q /content/drive/MyDrive/data.zip -d /content/dataset/
print("✅ 데이터 압축 해제 완료!")

Mounted at /content/drive
데이터 압축을 푸는 중입니다... (약 1~2분 소요)
✅ 데이터 압축 해제 완료!


In [2]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 디바이스: {device}")

NUM_CLASSES = 21

# 1. 빈 껍데기 모델 생성 (1채널, 21클래스 - 학습 때와 100% 동일한 구조)
model = models.resnet18()
model.conv1 = nn.Conv2d(in_channels=1,
                        out_channels=model.conv1.out_channels,
                        kernel_size=model.conv1.kernel_size,
                        stride=model.conv1.stride,
                        padding=model.conv1.padding,
                        bias=False)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# 2. 구글 드라이브에 저장된 가중치 불러오기
weights_path = '/content/drive/MyDrive/best_chart_resnet_model.pth'

# map_location=device를 넣으면 GPU가 없는 환경(CPU)에서도 에러 없이 로드됩니다.
model.load_state_dict(torch.load(weights_path, map_location=device))
model = model.to(device)

# 3. 평가(추론) 모드 전환 (매우 중요!)
model.eval()
print("✅ 모델 가중치 로드 완료! 똑똑한 모델이 부활했습니다.")

현재 디바이스: cpu
✅ 모델 가중치 로드 완료! 똑똑한 모델이 부활했습니다.


<h1>train cell 6</h1>


In [4]:
from PIL import Image
import torchvision.transforms as transforms
import torch.nn.functional as F

# 1. 클래스 이름 라벨
class_names = {
    0: "수평 채널", 1: "상승 채널", 2: "하락 채널", 3: "대칭 삼각수렴", 4: "상승 삼각수렴", 5: "하락 삼각수렴",
    6: "하락 쐐기형", 7: "상승 쐐기형", 8: "확장형", 9: "단일 지지/저항", 10: "헤드앤숄더", 11: "역헤드앤숄더",
    12: "쌍봉", 13: "쌍바닥", 14: "삼산", 15: "삼천", 16: "컵 앤 핸들", 17: "원형 바닥", 18: "상승 깃발형",
    19: "하락 깃발형", 20: "Other(노이즈)"
}

# ⭐️ 파이토치 사전순 꼬임 해결 통역기
folder_names = sorted([f"class_{i}" for i in range(21)])

# 2. 이미지 전처리 (학습 때와 100% 동일)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# 3. 예측 함수 (번역기 탑재)
def predict_stock_chart(model, image_tensor, threshold=0.85):
    with torch.no_grad():
        image_tensor = image_tensor.to(device).unsqueeze(0)
        outputs = model(image_tensor)
        probabilities = F.softmax(outputs, dim=1)

        # 모델이 뱉은 인덱스 확인
        max_prob, predicted_idx = torch.max(probabilities, 1)
        max_prob_value = max_prob.item()

        # 인덱스 -> 폴더명 -> 실제 클래스 번호로 번역
        predicted_folder_name = folder_names[predicted_idx.item()]
        original_pred_idx = int(predicted_folder_name.split('_')[1])

        # Threshold 방어 로직
        if max_prob_value < threshold:
            final_class = 20
        else:
            final_class = original_pred_idx

        return final_class, max_prob_value, original_pred_idx

# ==========================================
# 4. 직접 채점해 보기 (0번 ~ 19번 클래스, 각 5장씩만 테스트)
# ==========================================
print("🚀 채점을 시작합니다...\n")

for i in range(21):
    for j in range(1000, 1005): # 너무 기니까 각 패턴당 5장씩만 봅니다
        test_image_path = f'/content/dataset/synthetic_chart_images/class_{i}/img_{j:04d}.png'

        image = Image.open(test_image_path).convert('RGB')
        image_tensor = transform(image)

        print(f"📂 [테스트 파일: class_{i} / img_{j:04d}.png]")

        final_pred, confidence, original_pred = predict_stock_chart(model, image_tensor, threshold=0.85)

        print("-" * 50)
        print(f"✅ 실제 정답: [{class_names[i]}]")
        print(f"💡 AI의 예측: [{class_names[original_pred]}] (확신도: {confidence * 100:.2f}%)")

        if final_pred == 20:
            print(f"🛡️ 최종 판정: ➡️ [{class_names[final_pred]}] (확신도 미달로 필터링됨 ⚠️)")
        elif final_pred == i:
            print(f"🎯 최종 판정: ➡️ [{class_names[final_pred]}] (정답입니다! ⭕)")
        else:
            print(f"💥 최종 판정: ➡️ [{class_names[final_pred]}] (틀렸습니다! ❌)")
        print("=" * 50)

🚀 채점을 시작합니다...

📂 [테스트 파일: class_0 / img_1000.png]
--------------------------------------------------
✅ 실제 정답: [수평 채널]
💡 AI의 예측: [수평 채널] (확신도: 100.00%)
🎯 최종 판정: ➡️ [수평 채널] (정답입니다! ⭕)
📂 [테스트 파일: class_0 / img_1001.png]
--------------------------------------------------
✅ 실제 정답: [수평 채널]
💡 AI의 예측: [수평 채널] (확신도: 100.00%)
🎯 최종 판정: ➡️ [수평 채널] (정답입니다! ⭕)
📂 [테스트 파일: class_0 / img_1002.png]
--------------------------------------------------
✅ 실제 정답: [수평 채널]
💡 AI의 예측: [수평 채널] (확신도: 100.00%)
🎯 최종 판정: ➡️ [수평 채널] (정답입니다! ⭕)
📂 [테스트 파일: class_0 / img_1003.png]
--------------------------------------------------
✅ 실제 정답: [수평 채널]
💡 AI의 예측: [수평 채널] (확신도: 100.00%)
🎯 최종 판정: ➡️ [수평 채널] (정답입니다! ⭕)
📂 [테스트 파일: class_0 / img_1004.png]
--------------------------------------------------
✅ 실제 정답: [수평 채널]
💡 AI의 예측: [수평 채널] (확신도: 100.00%)
🎯 최종 판정: ➡️ [수평 채널] (정답입니다! ⭕)
📂 [테스트 파일: class_1 / img_1000.png]
--------------------------------------------------
✅ 실제 정답: [상승 채널]
💡 AI의 예측: [단일 지지/저항] (확신도: 51.79%)
🛡️ 최종 판정: ➡

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/synthetic_chart_images/class_20/img_1000.png'

In [9]:
import os

print("🚀 실제 존재하는 파일만 자동으로 찾아서 채점을 시작합니다...\n")

# 0번부터 20번(노이즈) 클래스까지 전부(21개) 검사
for i in range(20,21):
    folder_path = f'/content/dataset/synthetic_chart_images/class_{i}'

    # 만약 해당 폴더가 아예 없으면 다음 번호로 넘어감 (에러 방지)
    if not os.path.exists(folder_path):
        continue

    # 폴더 안에 있는 '실제 파일 이름'들을 리스트로 가져와서 정렬
    image_files = sorted(os.listdir(folder_path))

    # 폴더가 비어있으면 패스
    if len(image_files) == 0:
        continue

    # 각 클래스별로 딱 '앞에서부터 5장'만 테스트 (원하시면 5를 10이나 50으로 바꿔도 됩니다!)
    for file_name in image_files[4200:4500]:
        test_image_path = os.path.join(folder_path, file_name)

        # 이미지 불러오기 및 전처리
        image = Image.open(test_image_path).convert('RGB')
        image_tensor = transform(image)

        print(f"📂 [테스트 파일: class_{i} / {file_name}]")

        # 모델 예측
        final_pred, confidence, original_pred = predict_stock_chart(model, image_tensor, threshold=0.85)

        print("-" * 50)
        print(f"✅ 실제 정답: [{class_names[i]}]")
        print(f"💡 AI의 예측: [{class_names[original_pred]}] (확신도: {confidence * 100:.2f}%)")

        if final_pred == 20:
            print(f"🛡️ 최종 판정: ➡️ [{class_names[final_pred]}] (확신도 미달로 필터링됨 ⚠️)")
        elif final_pred == i:
            print(f"🎯 최종 판정: ➡️ [{class_names[final_pred]}] (정답입니다! ⭕)")
        else:
            print(f"💥 최종 판정: ➡️ [{class_names[final_pred]}] (틀렸습니다! ❌)")
        print("=" * 50)

🚀 실제 존재하는 파일만 자동으로 찾아서 채점을 시작합니다...

📂 [테스트 파일: class_20 / img_04200.png]
--------------------------------------------------
✅ 실제 정답: [Other(노이즈)]
💡 AI의 예측: [Other(노이즈)] (확신도: 100.00%)
🛡️ 최종 판정: ➡️ [Other(노이즈)] (확신도 미달로 필터링됨 ⚠️)
📂 [테스트 파일: class_20 / img_04201.png]
--------------------------------------------------
✅ 실제 정답: [Other(노이즈)]
💡 AI의 예측: [Other(노이즈)] (확신도: 100.00%)
🛡️ 최종 판정: ➡️ [Other(노이즈)] (확신도 미달로 필터링됨 ⚠️)
📂 [테스트 파일: class_20 / img_04202.png]
--------------------------------------------------
✅ 실제 정답: [Other(노이즈)]
💡 AI의 예측: [Other(노이즈)] (확신도: 100.00%)
🛡️ 최종 판정: ➡️ [Other(노이즈)] (확신도 미달로 필터링됨 ⚠️)
📂 [테스트 파일: class_20 / img_04203.png]
--------------------------------------------------
✅ 실제 정답: [Other(노이즈)]
💡 AI의 예측: [Other(노이즈)] (확신도: 100.00%)
🛡️ 최종 판정: ➡️ [Other(노이즈)] (확신도 미달로 필터링됨 ⚠️)
📂 [테스트 파일: class_20 / img_04204.png]
--------------------------------------------------
✅ 실제 정답: [Other(노이즈)]
💡 AI의 예측: [Other(노이즈)] (확신도: 100.00%)
🛡️ 최종 판정: ➡️ [Other(노이즈)] (확신도 미달로 필터링됨 ⚠️)
📂 [